## Setup

## claim_rejection_rate
**Tables:** `fact_claims` + `dim_policy` + `dim_region`

**Business metric** — rejection rate by policy CSL, policy state and customer region


In [0]:
from pyspark.sql.functions import (
    col, count, round as spark_round,
    sum as spark_sum, current_timestamp, lit, when
)

fact_claims = spark.table("primeinsurance.gold.fact_claims")
dim_policy  = spark.table("primeinsurance.gold.dim_policy")
dim_region  = spark.table("primeinsurance.gold.dim_region")

claim_rejection_rate = (
    fact_claims

    # Join dim_policy for policy details
    .join(
        dim_policy.select(
            col("policy_number"),
            col("policy_csl"),
            col("policy_deductable"),
            col("policy_annual_premium")
        ),
        fact_claims["policy_id"] == dim_policy["policy_number"],
        "left"
    )

    # Join dim_region for region details
    .join(
        dim_region,
        fact_claims["region"] == dim_region["region_name"],
        "left"
    )

    # Group by region, policy state, policy CSL
    .groupBy(
        col("region_name").alias("region"),
        col("policy_state"),
        dim_policy["policy_csl"]
    )
    .agg(
        count("claim_id").alias("total_claims"),
        spark_sum(when(col("claim_rejected") == "Y", 1).otherwise(0)).alias("rejected_claims"),
        spark_sum(when(col("claim_rejected") == "N", 1).otherwise(0)).alias("approved_claims"),
        spark_round(
            spark_sum(when(col("claim_rejected") == "Y", 1).otherwise(0)) * 100.0 /
            count("claim_id"), 2
        ).alias("rejection_rate_pct")
    )
    .orderBy("rejection_rate_pct", ascending=False)
    .withColumn("gold_created_at", current_timestamp())
)

(
    claim_rejection_rate.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.claim_rejection_rate")
)

print(f"gold.claim_rejection_rate : {claim_rejection_rate.count():,} rows")
spark.table("primeinsurance.gold.claim_rejection_rate").display()

## avg_processing_time
**Table:** `fact_claims`

**Performance metric** — claim processing status breakdown by incident type and severity

> ⚠️ Date fields (`Claim_Logged_On`, `Claim_Processed_On`) are corrupted in source — actual time difference cannot be calculated. Metric shows processed vs unprocessed count and rate instead.


In [0]:
from pyspark.sql.functions import (
    col, count, round as spark_round,
    sum as spark_sum, current_timestamp, when
)

fact_claims = spark.table("primeinsurance.gold.fact_claims")

avg_processing_time = (
    fact_claims
    .groupBy(
        col("incident_type"),
        col("incident_severity")
    )
    .agg(
        count("claim_id").alias("total_claims"),

        # Processed = Claim_Processed_On is not null
        spark_sum(
            when(col("claim_processed_on").isNotNull(), 1).otherwise(0)
        ).alias("processed_claims"),

        # Unprocessed = Claim_Processed_On is null
        spark_sum(
            when(col("claim_processed_on").isNull(), 1).otherwise(0)
        ).alias("unprocessed_claims"),

        # Processing rate %
        spark_round(
            spark_sum(when(col("claim_processed_on").isNotNull(), 1).otherwise(0)) * 100.0 /
            count("claim_id"), 2
        ).alias("processing_rate_pct"),

        # Approved vs rejected breakdown
        spark_sum(when(col("claim_rejected") == "N", 1).otherwise(0)).alias("approved_claims"),
        spark_sum(when(col("claim_rejected") == "Y", 1).otherwise(0)).alias("rejected_claims")
    )
    .orderBy("total_claims", ascending=False)
    .withColumn("gold_created_at", current_timestamp())
)

(
    avg_processing_time.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.avg_processing_time")
)

print(f" gold.avg_processing_time  : {avg_processing_time.count():,} rows")
spark.table("primeinsurance.gold.avg_processing_time").display()


## unsold_cars
**Tables:** `fact_sales` + `dim_car`

**Inventory metric** — cars listed but not yet sold, broken down by model, fuel, transmission


In [0]:
from pyspark.sql.functions import (
    col, count, round as spark_round,
    sum as spark_sum, current_timestamp, when
)

fact_sales = spark.table("primeinsurance.gold.fact_sales")
dim_car    = spark.table("primeinsurance.gold.dim_car")

unsold_cars = (
    fact_sales
    # Select only columns unique to fact_sales to avoid ambiguity
    .select(
        col("car_id"),
        col("sales_id"),
        col("sold_on"),
        col("seller_type"),
        col("Region")
    )

    # Join dim_car for full car details
    .join(
        dim_car.select(
            col("car_id"),
            col("name").alias("car_name"),
            col("model"),
            col("fuel"),
            col("transmission"),
            col("km_driven"),
            col("mileage"),
            col("seats")
        ),
        "car_id",
        "left"
    )

    # Group by car attributes
    .groupBy(
        col("car_id"),
        col("car_name"),
        col("model"),
        col("fuel"),
        col("transmission"),
        col("km_driven"),
        col("mileage"),
        col("seats"),
        col("seller_type"),
        col("Region")
    )
    .agg(
        count("sales_id").alias("total_listings"),
        spark_sum(when(col("sold_on").isNull(),    1).otherwise(0)).alias("unsold_count"),
        spark_sum(when(col("sold_on").isNotNull(), 1).otherwise(0)).alias("sold_count"),
        spark_round(
            spark_sum(when(col("sold_on").isNull(), 1).otherwise(0)) * 100.0 /
            count("sales_id"), 2
        ).alias("unsold_rate_pct")
    )

    # Filter only cars that have at least 1 unsold listing
    .filter(col("unsold_count") > 0)
    .orderBy("unsold_count", ascending=False)
    .withColumn("gold_created_at", current_timestamp())
)

(
    unsold_cars.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.unsold_cars")
)

print(f" gold.unsold_cars          : {unsold_cars.count():,} rows")
spark.table("primeinsurance.gold.unsold_cars").display()

## customer_count
**Tables:** `dim_customer` + `dim_region`

**Analytics metric** — customer distribution by region, education, marital status and financial profile


In [0]:
from pyspark.sql.functions import (
    col, count, round as spark_round,
    sum as spark_sum, avg as spark_avg,
    current_timestamp, when
)

dim_customer = spark.table("primeinsurance.gold.dim_customer")
dim_region   = spark.table("primeinsurance.gold.dim_region")

customer_count = (
    dim_customer

    # Join dim_region for region_id
    .join(
        dim_region,
        dim_customer["region"] == dim_region["region_name"],
        "left"
    )

    # Group by region, education, marital status
    .groupBy(
        col("region_name").alias("region"),
        col("education"),
        col("marital_status")
    )
    .agg(
        count("customer_id").alias("customer_count"),

        # Financial profile
        spark_round(spark_avg("balance"), 2).alias("avg_balance"),
        spark_sum(when(col("default")      == 1, 1).otherwise(0)).alias("defaulted_customers"),
        spark_sum(when(col("hh_insurance") == 1, 1).otherwise(0)).alias("hh_insured_customers"),
        spark_sum(when(col("car_loan")     == 1, 1).otherwise(0)).alias("car_loan_customers"),

        # Default rate %
        spark_round(
            spark_sum(when(col("default") == 1, 1).otherwise(0)) * 100.0 /
            count("customer_id"), 2
        ).alias("default_rate_pct")
    )
    .orderBy("customer_count", ascending=False)
    .withColumn("gold_created_at", current_timestamp())
)

(
    customer_count.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("primeinsurance.gold.customer_count")
)

print(f" gold.customer_count       : {customer_count.count():,} rows")
spark.table("primeinsurance.gold.customer_count").display()


## Summary

In [0]:
agg_tables = [
    "claim_rejection_rate",
    "avg_processing_time",
    "unsold_cars",
    "customer_count"
]

print("=" * 55)
print(f"{'Aggregation':<25} {'Rows':>8} {'Columns':>8}")
print("=" * 55)
for tbl in agg_tables:
    df = spark.table(f"primeinsurance.gold.{tbl}")
    print(f"{tbl:<25} {df.count():>8,} {len(df.columns):>8}")
print("=" * 55)
print("\n All aggregations complete — primeinsurance.gold")
